In [1]:
import anthropic     # connects your code to Claude
import pandas as pd  # reads CSV, handles data tables
import json          # parses Claude's JSON response
import os            # creates folders, handles paths


In [ ]:
from groq import Groq

client = Groq(api_key="api-key")

print("Connected to Groq")

Connected to Groq


In [21]:
# Change this:
df_raw = pd.read_csv("WEO_Data.csv", encoding="latin-1")

# To this — add sep="\t":
df_raw = pd.read_csv("WEO_Data.csv", encoding="latin-1", sep="\t")

print("Shape:", df_raw.shape)
print("\nColumns:", list(df_raw.columns)[:10])
print("\nFirst few rows:")
print(df_raw.head(3))

Shape: (7568, 16)

Columns: ['WEO Country Code', 'WEO Subject Code', 'Country', 'Subject Descriptor', 'Units', 'Scale', 'Country/Series-specific Notes', '2020', '2021', '2022']

First few rows:
  WEO Country Code WEO Subject Code      Country  \
0              512           NGDP_R  Afghanistan   
1              512        NGDP_RPCH  Afghanistan   
2              512             NGDP  Afghanistan   

                        Subject Descriptor              Units     Scale  \
0  Gross domestic product, constant prices  National currency  Billions   
1  Gross domestic product, constant prices     Percent change     Units   
2   Gross domestic product, current prices  National currency  Billions   

                       Country/Series-specific Notes       2020       2021  \
0  Source: National Statistics Office Latest actu...  1,288.869  1,101.445   
1  See notes for:  Gross domestic product, consta...     -2.351    -14.542   
2  Source: National Statistics Office Latest actu...  1,547.28

In [22]:
# Fix column name — strip any trailing whitespace
df_raw.columns = df_raw.columns.str.strip()

# The indicators we want — copy exact names from the WEO file
INDICATORS = [
    "Gross domestic product, constant prices",
    "Gross domestic product, current prices",
    "Inflation, average consumer prices",
    "Unemployment rate",
    "General government net lending/borrowing",
    "Current account balance",
    "Volume of imports of goods and services",
    "Volume of exports of goods and services"
]

COUNTRIES = [
    "United Kingdom",
    "Brazil",
    "Nigeria",
    "Egypt",
    "Indonesia",
    "Argentina"
]

# Filter rows
df_filtered = df_raw[
    df_raw["Country"].isin(COUNTRIES) &
    df_raw["Subject Descriptor"].isin(INDICATORS)
].copy()

# Check what we got
print("Rows found:", len(df_filtered))
print("Countries found:", df_filtered["Country"].unique())
print("Indicators found:", df_filtered["Subject Descriptor"].unique())

Rows found: 81
Countries found: ['Argentina' 'Brazil' 'Egypt' 'Indonesia' 'Nigeria' 'United Kingdom']
Indicators found: ['Gross domestic product, constant prices'
 'Gross domestic product, current prices'
 'Inflation, average consumer prices'
 'Volume of imports of goods and services'
 'Volume of exports of goods and services' 'Unemployment rate'
 'General government net lending/borrowing' 'Current account balance']


In [23]:
# Select only the columns we need
df_clean = df_filtered[["Country", "Subject Descriptor", "Units", "2024"]].copy()

# Rename for clarity
df_clean.columns = ["country", "indicator", "units", "value_2024"]

# Remove commas from numbers like "1,288.869" so pandas can read them as numbers
df_clean["value_2024"] = df_clean["value_2024"].astype(str).str.replace(",", "", regex=False)

# Convert to numeric — anything that can't convert becomes NaN
df_clean["value_2024"] = pd.to_numeric(df_clean["value_2024"], errors="coerce")

# Drop rows where 2024 value is missing
df_clean = df_clean.dropna(subset=["value_2024"])

print(f"Clean data: {len(df_clean)} rows remaining")
print()
print(df_clean.to_string(index=False))

Clean data: 81 rows remaining

       country                                indicator                                          units    value_2024
     Argentina  Gross domestic product, constant prices                              National currency  7.021810e+02
     Argentina  Gross domestic product, constant prices                                 Percent change -1.719000e+00
     Argentina   Gross domestic product, current prices                              National currency  5.792458e+05
     Argentina   Gross domestic product, current prices                                   U.S. dollars  6.321450e+02
     Argentina   Gross domestic product, current prices Purchasing power parity; international dollars  1.378908e+03
     Argentina       Inflation, average consumer prices                                          Index  1.183509e+04
     Argentina       Inflation, average consumer prices                                 Percent change  2.198850e+02
     Argentina  Volume of imports

In [24]:
# For each indicator, keep only the most useful unit
# Percent change is most useful for growth/inflation/trade
# Percent of GDP is most useful for fiscal/current account
# Percent of total labor force for unemployment

UNIT_PRIORITY = {
    "Gross domestic product, constant prices": "Percent change",
    "Gross domestic product, current prices": "U.S. dollars",
    "Inflation, average consumer prices": "Percent change",
    "Volume of imports of goods and services": "Percent change",
    "Volume of exports of goods and services": "Percent change",
    "Unemployment rate": "Percent of total labor force",
    "General government net lending/borrowing": "Percent of GDP",
    "Current account balance": "Percent of GDP"
}

# Filter to keep only the priority unit per indicator
rows_to_keep = []

for indicator, preferred_unit in UNIT_PRIORITY.items():
    mask = (df_clean["indicator"] == indicator) & (df_clean["units"] == preferred_unit)
    rows_to_keep.append(df_clean[mask])

df_final = pd.concat(rows_to_keep, ignore_index=True)

print(f" {len(df_final)} rows — clean and deduplicated")
print()
print(df_final.to_string(index=False))

 47 rows — clean and deduplicated

       country                                indicator                        units  value_2024
     Argentina  Gross domestic product, constant prices               Percent change      -1.719
        Brazil  Gross domestic product, constant prices               Percent change       3.396
         Egypt  Gross domestic product, constant prices               Percent change       2.399
     Indonesia  Gross domestic product, constant prices               Percent change       5.030
       Nigeria  Gross domestic product, constant prices               Percent change       3.426
United Kingdom  Gross domestic product, constant prices               Percent change       1.101
     Argentina   Gross domestic product, current prices                 U.S. dollars     632.145
        Brazil   Gross domestic product, current prices                 U.S. dollars    2171.337
         Egypt   Gross domestic product, current prices                 U.S. dollars     383

In [25]:
def format_country_data(country_name, df_final):
    """
    Formats one country's data into readable text for Claude.
    """
    country_data = df_final[df_final["country"] == country_name]
    
    lines = [f"IMF World Economic Outlook 2024 estimates for {country_name}:"]
    
    for _, row in country_data.iterrows():
        lines.append(f"- {row['indicator']} ({row['units']}): {round(row['value_2024'], 2)}")
    
    return "\n".join(lines)


# Preview what Claude will receive for one country
print(format_country_data("Argentina", df_final))


IMF World Economic Outlook 2024 estimates for Argentina:
- Gross domestic product, constant prices (Percent change): -1.72
- Gross domestic product, current prices (U.S. dollars): 632.14
- Inflation, average consumer prices (Percent change): 219.88
- Volume of imports of goods and services (Percent change): -23.47
- Volume of exports of goods and services (Percent change): 29.14
- Unemployment rate (Percent of total labor force): 7.15
- General government net lending/borrowing (Percent of GDP): 0.86
- Current account balance (Percent of GDP): 0.99


In [26]:
def analyse_country_risk(country_name, df_final, client):
    
    country_data_text = format_country_data(country_name, df_final)
    
    prompt = f"""You are a senior country risk analyst at an investment bank.

Based on the following IMF 2024 economic indicators for {country_name}, 
produce a structured risk assessment.

{country_data_text}

Return ONLY a valid JSON object. No explanation, no markdown, no text before or after.
Exactly this structure:

{{
    "country": "{country_name}",
    "overall_risk_level": "Low or Medium or High or Critical",
    "risk_categories": {{
        "economic": {{
            "score": <integer 1 to 10 where 10 is highest risk>,
            "summary": "2 clear sentences on economic risk based on the data",
            "key_signals": ["specific data point as signal", "signal 2", "signal 3"]
        }},
        "financial": {{
            "score": <integer 1 to 10>,
            "summary": "2 clear sentences on financial stability risk",
            "key_signals": ["signal 1", "signal 2", "signal 3"]
        }},
        "political": {{
            "score": <integer 1 to 10>,
            "summary": "2 sentences on political risk inferred from economic indicators",
            "key_signals": ["signal 1", "signal 2", "signal 3"]
        }},
        "operational": {{
            "score": <integer 1 to 10>,
            "summary": "2 sentences on operational and business environment risk",
            "key_signals": ["signal 1", "signal 2", "signal 3"]
        }}
    }},
    "executive_summary": "3 sentence narrative summarising overall risk for a senior stakeholder",
    "data_quality_flag": "Note any indicators that are missing, extreme, or potentially unreliable. Write No issues detected if all looks clean.",
    "recommended_action": "Monitor or Caution or Restrict or Avoid"
}}
"""

    # ← ONLY THIS BLOCK CHANGED
    response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ],
    max_tokens=1500 
    )
    raw_response = response.choices[0].message.content
    # ← END OF CHANGED BLOCK

    try:
        return json.loads(raw_response)
    except json.JSONDecodeError:
        start = raw_response.find('{')
        end = raw_response.rfind('}') + 1
        return json.loads(raw_response[start:end])


# Test on Argentina
test_result = analyse_country_risk("Argentina", df_final, client)
print(json.dumps(test_result, indent=2))

{
  "country": "Argentina",
  "overall_risk_level": "High",
  "risk_categories": {
    "economic": {
      "score": 8,
      "summary": "Argentina's economy is experiencing a contraction, with a -1.72% change in GDP, and an extremely high inflation rate of 219.88%, indicating significant economic instability. The decline in imports and increase in exports suggest a potential trade imbalance and challenges in meeting domestic demand.",
      "key_signals": [
        "-1.72% GDP growth",
        "219.88% inflation rate",
        "-23.47% decline in imports"
      ]
    },
    "financial": {
      "score": 7,
      "summary": "The general government net lending/borrowing ratio is relatively stable at 0.86% of GDP, but the high inflation rate poses a risk to financial stability and the ability to manage debt. The current account balance is slightly positive, but the overall financial situation is precarious due to the economic contraction and high inflation.",
      "key_signals": [
      

In [27]:
COUNTRIES_TO_ANALYSE = [
    "United Kingdom",
    "Brazil",
    "Nigeria",
    "Egypt",
    "India",
    "Germany",
    "Indonesia",
    "Argentina",
    "Switzerland",
    "Singapore",
    "Australia",
    "Scandinavia"


]

all_results = []

for country in COUNTRIES_TO_ANALYSE:
    result = analyse_country_risk(country, df_final, client)
    all_results.append(result)
    print(f"{country} — {result['overall_risk_level']} | Action: {result['recommended_action']}")

print(f"\n🎯 All {len(all_results)} countries analysed")

United Kingdom — Medium | Action: Monitor
Brazil — Medium | Action: Monitor
Nigeria — High | Action: Caution
Egypt — High | Action: Caution
India — Medium | Action: Monitor
Germany — Medium | Action: Monitor
Indonesia — Medium | Action: Monitor
Argentina — High | Action: Caution
Switzerland — Low | Action: Monitor
Singapore — Low | Action: Monitor
Australia — Medium | Action: Monitor
Scandinavia — Medium | Action: Monitor

🎯 All 12 countries analysed


In [28]:
rows = []

for result in all_results:
    for category, data in result["risk_categories"].items():
        rows.append({
            "country": result["country"],
            "overall_risk_level": result["overall_risk_level"],
            "recommended_action": result["recommended_action"],
            "risk_category": category,
            "risk_score": data["score"],
            "summary": data["summary"],
            "key_signal_1": data["key_signals"][0] if len(data["key_signals"]) > 0 else "",
            "key_signal_2": data["key_signals"][1] if len(data["key_signals"]) > 1 else "",
            "key_signal_3": data["key_signals"][2] if len(data["key_signals"]) > 2 else "",
            "executive_summary": result["executive_summary"],
            "data_quality_flag": result["data_quality_flag"]
        })

df_output = pd.DataFrame(rows)

os.makedirs("output", exist_ok=True)
df_output.to_csv("output/country_risk_scores.csv", index=False)

print(f"CSV saved — {len(df_output)} rows")
print()
print(df_output[["country", "risk_category", "risk_score", "overall_risk_level", "recommended_action"]].to_string(index=False))

CSV saved — 48 rows

       country risk_category  risk_score overall_risk_level recommended_action
United Kingdom      economic           6             Medium            Monitor
United Kingdom     financial           5             Medium            Monitor
United Kingdom     political           4             Medium            Monitor
United Kingdom   operational           5             Medium            Monitor
        Brazil      economic           6             Medium            Monitor
        Brazil     financial           7             Medium            Monitor
        Brazil     political           5             Medium            Monitor
        Brazil   operational           4             Medium            Monitor
       Nigeria      economic           8               High            Caution
       Nigeria     financial           6               High            Caution
       Nigeria     political           7               High            Caution
       Nigeria   operational   